# Notebook 1 -- LLM vote tables (data production + accuracy proxies)

**Context.** Three LLMs read transcripts of 191 ONUW games (public dialogue +
rules only, no private information) and vote for the Werewolf (or "No
Werewolf"), with a brief justification. The thesis question is how much the
vote reflects **deduction from game facts** vs. **social pressure /
persuasion** in the dialogue. Part 1 analyses votes; part 2 analyses
justifications. This notebook is the *data producer* for part 1: it parses the
raw run outputs, evaluates them, and writes the tables everything downstream
reads. Accuracy appears here only as a **proxy** -- it is deliberately not the
final metric (werewolves can be persuasive; the transcript can be genuinely
underdetermined).

**Data contract -- files this notebook produces:**

Model-specific, under `analysis/<LLM_NAME>/<MODEL_STAGE>/voting/<prompt>/vote_stability/tables/`:

| file | grain | content |
|---|---|---|
| `llm_vote_file_level.csv` | one row per (run, game) | parsed vote, status, justification, end **and start** role of the voted player |
| `llm_vote_game_level.csv` | one row per game | stochastic vote distribution, `p_correct`, entropy, majority diagnostics, greedy vote |
| `llm_run_summary.csv` | one row per run | accuracy proxies and status counts |

Model-specific printable examples under `.../vote_stability/examples/`.

Human outcomes are **model-agnostic**, so they are written once, outside any
model folder, under `analysis/human_outcomes/<prompt>/tables/`:

| file | grain | content |
|---|---|---|
| `human_game_outcomes.csv` | one row per game | mechanical table outcome from all human votes |
| `village_vote_dispersion.csv` | one row per game | village-aligned-only vote counts, modal target, entropy (same schema as before, new location) |
| `human_summary.csv` | one row | corpus-level human baseline |

**Aggregating the 3 stochastic runs.** The stochastic runs are sampled at
T = 1, i.e. from the model's *unmodified* output distribution. Three runs are
therefore three i.i.d. draws from p_model(vote | transcript), and the honest
aggregate is the **empirical distribution** over those draws -- not a majority
label. Per game the primary quantity is `p_correct`: the empirical probability
mass on the correct answer, an unbiased estimate of the probability that a
single T=1 sample is correct. Its mean over games equals mean single-run
stochastic accuracy (checked below). Majority vote and "at least one correct"
are kept as secondary diagnostics. The **greedy** run is a different decoding
mode (approximate argmax), so it is always reported as its own column and never
pooled with the stochastic draws.


## Configuration

In [12]:
from pathlib import Path
from typing import Any, Optional
from collections import Counter
from itertools import combinations
import json
import re

import numpy as np
import pandas as pd

REPO_NAME = "masters_thesis_sdg"

# Input voting JSONs are expected under exactly:
#   results/voting/<LLM_NAME>/prompt_v4_run_{1,2,3}   (stochastic, T=1)
#   results/voting/<LLM_NAME>/prompt_v4_t0            (greedy)
LLM_NAME = "unsloth_gemma-4-E2B-it-unsloth-bnb-4bit"
MODEL_STAGE = "base"
PROMPT_FAMILY = "v4"

EXPECTED_STOCHASTIC_RUNS = (1, 2, 3)
EXPECT_GREEDY_RUN = True
STOCHASTIC_TEMPERATURE = 1.0

CIRCLE_LABELS = {"no werewolf"}
CIRCLE_VOTE_STRING = "No Werewolf"   # canonical label used in vote distributions

# Roles whose holders are NOT village-aligned when computing the human
# village-only dispersion. Alignment is taken from ALIGNMENT_ROLES_FROM roles:
# "end" matches the win condition; "start" matches what each voter believed
# during the dialogue. Keep "end" unless you have a reason to switch.
NON_VILLAGE_ALIGNED_ROLES = {"Werewolf", "Minion", "Tanner"}
ALIGNMENT_ROLES_FROM = "end"

RANDOM_STATE = 42
SAMPLE_PER_EXAMPLE_TYPE = 3

SAVE_TABLES = True
SAVE_PRINTABLE_EXAMPLES = True


## Utilities and paths

In [13]:
def strip_prompt_prefix(prompt_family: str) -> str:
    return str(prompt_family).removeprefix("prompt_")


def find_repo_root(start: Optional[Path] = None, repo_name: str = REPO_NAME) -> Path:
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Could not find repo root {repo_name!r}")
        current = current.parent


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_csv(df: pd.DataFrame, path: Path, json_columns: tuple = ()) -> None:
    """Save a dataframe, JSON-encoding list/dict columns so they round-trip."""
    out = df.copy()
    for col in json_columns:
        if col in out.columns:
            out[col] = out[col].apply(lambda v: json.dumps(v) if isinstance(v, (list, dict)) else v)
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, index=False)
    print(f"saved {len(out):>5} rows -> {path.relative_to(REPO_ROOT)}")


PROMPT_DIR_PREFIX = f"prompt_{strip_prompt_prefix(PROMPT_FAMILY)}"

REPO_ROOT = find_repo_root()
IDX_FILE_PATH = REPO_ROOT / "data" / "processed" / "lai2023" / "onuw_transcripts_ready" / "index_cleaned.json"
RUNS_ROOT = REPO_ROOT / "results" / "voting" / LLM_NAME

MODEL_TABLES_DIR = (REPO_ROOT / "analysis" / LLM_NAME / MODEL_STAGE / "voting"
                    / PROMPT_DIR_PREFIX / "vote_stability" / "tables")
MODEL_EXAMPLES_DIR = MODEL_TABLES_DIR.parent / "examples"
HUMAN_TABLES_DIR = REPO_ROOT / "analysis" / "human_outcomes" / PROMPT_DIR_PREFIX / "tables"

print("REPO_ROOT :", REPO_ROOT)
print("RUNS_ROOT :", RUNS_ROOT)
print("model tables ->", MODEL_TABLES_DIR.relative_to(REPO_ROOT))
print("human tables ->", HUMAN_TABLES_DIR.relative_to(REPO_ROOT))


REPO_ROOT : C:\Users\annab\Documents\GitHub\masters_thesis_sdg
RUNS_ROOT : C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit
model tables -> analysis\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables
human tables -> analysis\human_outcomes\prompt_v4\tables


## Ground truth

One row per game from `index_cleaned.json`. Correctness is defined on **end
roles** (the win condition): a player vote is correct iff the target is an
end-Werewolf; a circle vote is correct iff no end-Werewolf exists. Start roles
are carried through so downstream analyses can also ask whether a vote hit a
player who *behaved* like a Werewolf during the dialogue (they started as one)
regardless of night swaps.


In [14]:
def normalize_missing_vote(vote: Any) -> bool:
    if vote is None:
        return True
    if isinstance(vote, str):
        return vote.strip().upper() in {"", "NA", "N/A", "NONE", "NULL"}
    return False


def parse_indexed_votes(raw_votes: Any) -> list:
    """Parse human vote indices, preserving voter position (None = missing)."""
    if not isinstance(raw_votes, list):
        return []
    parsed = []
    for vote in raw_votes:
        if normalize_missing_vote(vote):
            parsed.append(None)
            continue
        try:
            parsed.append(int(vote))
        except (TypeError, ValueError):
            parsed.append(None)
    return parsed


def build_ground_truth_df(index_data: list) -> pd.DataFrame:
    rows = []
    for game in index_data:
        source, session_name, game_key = game["source"], game["session_name"], game["game_key"]
        processed_txt_path = game.get("processed_txt_path")
        game_name = Path(processed_txt_path).stem if processed_txt_path else f"{session_name}_{game_key}"

        player_names = game["player_names"]
        start_roles = game["start_roles"]
        end_roles = game["end_roles"]

        correct_player_indices = [i for i, role in enumerate(end_roles) if role == "Werewolf"]
        rows.append({
            "game_id": f"{source} / {session_name} / {game_key}",
            "game_name": game_name,
            "source": source, "session_name": session_name, "game_key": game_key,
            "processed_txt_path": processed_txt_path,
            "num_players": game.get("num_players"), "num_turns": game.get("num_turns"),
            "player_names": player_names, "start_roles": start_roles, "end_roles": end_roles,
            "has_werewolf": len(correct_player_indices) > 0,
            "n_werewolves": len(correct_player_indices),
            "correct_player_indices": correct_player_indices,
            "correct_player_names": [player_names[i] for i in correct_player_indices],
            "correct_vote_type": "player" if correct_player_indices else "circle",
            "voting_outcome": game.get("voting_outcome", []),
            "warning": game.get("warning", []),
        })
    return pd.DataFrame(rows)


index_data = load_json(IDX_FILE_PATH)
gt_df = build_ground_truth_df(index_data)
print(f"Ground-truth games: {len(gt_df)} ({gt_df['game_name'].nunique()} unique game names)")
display(gt_df["correct_vote_type"].value_counts().to_frame("n_games"))
display(gt_df["n_werewolves"].value_counts().sort_index().to_frame("n_games"))


Ground-truth games: 191 (191 unique game names)


,n_games
correct_vote_type,
player,165
circle,26


,n_games
n_werewolves,
0,26
1,101
2,64


## Discover run directories

In [15]:
prompt_run_dirs = [
    {
        "prompt_dir": RUNS_ROOT / f"{PROMPT_DIR_PREFIX}_run_{i}",
        "prompt_version": f"{PROMPT_DIR_PREFIX}_run_{i}",
        "decoding": "stochastic", "temperature": STOCHASTIC_TEMPERATURE,
        "run_number": i, "run_label": f"run_{i}",
    }
    for i in EXPECTED_STOCHASTIC_RUNS
]
if EXPECT_GREEDY_RUN:
    prompt_run_dirs.append({
        "prompt_dir": RUNS_ROOT / f"{PROMPT_DIR_PREFIX}_t0",
        "prompt_version": f"{PROMPT_DIR_PREFIX}_t0",
        "decoding": "greedy", "temperature": 0.0,
        "run_number": 0, "run_label": "greedy_t0",
    })

STOCHASTIC_RUN_LABELS = [r["run_label"] for r in prompt_run_dirs if r["decoding"] == "stochastic"]
GREEDY_RUN_LABEL = "greedy_t0"

run_dirs_df = pd.DataFrame([
    {"run_label": r["run_label"], "decoding": r["decoding"], "temperature": r["temperature"],
     "prompt_dir": str(r["prompt_dir"].relative_to(REPO_ROOT)), "exists": r["prompt_dir"].exists()}
    for r in prompt_run_dirs
])
display(run_dirs_df)
missing = run_dirs_df[~run_dirs_df["exists"]]
if not missing.empty:
    print("WARNING: missing run directories -- results will be partial:")
    print(missing["run_label"].tolist())


,run_label,decoding,temperature,prompt_dir,exists
0,run_1,stochastic,1.0,results\voting\unsloth_gemma-4-E2B-it-unsloth-...,True
1,run_2,stochastic,1.0,results\voting\unsloth_gemma-4-E2B-it-unsloth-...,True
2,run_3,stochastic,1.0,results\voting\unsloth_gemma-4-E2B-it-unsloth-...,True
3,greedy_t0,greedy,0.0,results\voting\unsloth_gemma-4-E2B-it-unsloth-...,True


## Parse and evaluate result files (file level)

Reuses the established parsing/matching logic. Additions relative to the old
table: `start_roles` and `voted_player_start_role` (appended as the last two
columns so existing readers are unaffected).


In [16]:
def normalize_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def is_circle_vote_label(vote: Any) -> bool:
    text = normalize_text(vote)
    return text.lower() in CIRCLE_LABELS if text is not None else False


def match_player_name(vote: Any, player_names: list) -> Optional[str]:
    vote_text = normalize_text(vote)
    if vote_text is None:
        return None
    if vote_text in player_names:
        return vote_text
    vote_norm = vote_text.lower()
    for player in player_names:
        if str(player).strip().lower() == vote_norm:
            return player
    return None


GT_BY_KEY = {
    (row["source"], row["session_name"], row["game_key"]): row
    for _, row in gt_df.iterrows()
}


def resolve_gt_row(res_data: dict) -> Optional[pd.Series]:
    # Match a result file to its game by (source, session, game) only.
    return GT_BY_KEY.get((res_data.get("source"), res_data.get("session_name"), res_data.get("game_key")))


def extract_model_vote(res_data: dict) -> tuple:
    # Read the vote and justification from the two fields the pipeline writes.
    validation = res_data.get("validation") or {}
    parsed_output = res_data.get("parsed_output") or {}
    chosen_vote = normalize_text(validation.get("chosen_vote"))
    is_valid = bool(validation.get("is_valid", chosen_vote is not None))
    justification = parsed_output.get("justification")
    return chosen_vote, is_valid, justification


In [17]:
INVALID_FIELDS = {
    "chosen_player_name": None, "is_circle_vote": False, "is_correct": np.nan,
    "voted_player_end_role": None, "voted_player_start_role": None,
}


def evaluate_one_result_file(res_file: Path, run_info: dict) -> dict:
    res_data = load_json(res_file)
    gt_row = resolve_gt_row(res_data)
    chosen_vote_raw, is_valid_parse, justification = extract_model_vote(res_data)

    base = {
        "path": str(res_file), "run_label": run_info["run_label"], "decoding": run_info["decoding"],
        "source": res_data.get("source"), "session_name": res_data.get("session_name"),
        "game_key": res_data.get("game_key"),
        "game_id": f"{res_data.get('source')} / {res_data.get('session_name')} / {res_data.get('game_key')}",
        "chosen_vote_raw": chosen_vote_raw, "justification": justification,
    }

    if gt_row is None:
        return {**base, "status": "missing_ground_truth", **INVALID_FIELDS}

    player_names, end_roles, start_roles = gt_row["player_names"], gt_row["end_roles"], gt_row["start_roles"]

    if not is_valid_parse or chosen_vote_raw is None:
        return {**base, "status": "failed_parse", **INVALID_FIELDS}

    matched = match_player_name(chosen_vote_raw, player_names)
    if matched is not None:
        idx = player_names.index(matched)
        return {**base, "status": "player_vote", "chosen_player_name": matched, "is_circle_vote": False,
                "is_correct": bool(end_roles[idx] == "Werewolf"),
                "voted_player_end_role": end_roles[idx],
                "voted_player_start_role": start_roles[idx] if idx < len(start_roles) else None}
    if is_circle_vote_label(chosen_vote_raw):
        return {**base, "status": "circle_vote", "chosen_player_name": None, "is_circle_vote": True,
                "is_correct": bool(not gt_row["has_werewolf"]),
                "voted_player_end_role": None, "voted_player_start_role": None}
    return {**base, "status": "invalid_vote", **INVALID_FIELDS}


rows = []
for run_info in prompt_run_dirs:
    result_files = sorted(run_info["prompt_dir"].rglob("*.json")) if run_info["prompt_dir"].exists() else []
    print(f"{run_info['run_label']:>10} | {run_info['decoding']:>10} | {len(result_files)} files")
    for res_file in result_files:
        rows.append(evaluate_one_result_file(res_file, run_info))
llm_vote_df = pd.DataFrame(rows)

unmatched = llm_vote_df[llm_vote_df["status"] == "missing_ground_truth"]
if len(unmatched):
    print()
    print(f"WARNING: {len(unmatched)} result file(s) did not match any game by (source, session, game):")
    for p in unmatched["path"].head(10):
        print("  ", p)

print()
display(llm_vote_df.groupby(["run_label", "status"]).size().unstack(fill_value=0))


     run_1 | stochastic | 191 files
     run_2 | stochastic | 191 files
     run_3 | stochastic | 191 files
 greedy_t0 |     greedy | 191 files



status,circle_vote,failed_parse,player_vote
run_label,,,
greedy_t0,48,0,143
run_1,39,1,151
run_2,43,2,146
run_3,49,2,140


In [18]:
# File-level table: one row per (run, game), vote-level facts only.
# Game-level facts (roster, roles, correct answer) live once per game in
# llm_vote_game_level.csv -- join on game_id when both are needed.
FILE_LEVEL_COLUMNS = [
    "path", "run_label", "decoding", "source", "session_name", "game_key", "game_id",
    "status", "chosen_vote_raw", "chosen_player_name", "is_circle_vote", "is_correct",
    "voted_player_end_role", "voted_player_start_role", "justification",
]
llm_vote_file_level = llm_vote_df.reindex(columns=FILE_LEVEL_COLUMNS)

if SAVE_TABLES:
    save_csv(llm_vote_file_level, MODEL_TABLES_DIR / "llm_vote_file_level.csv")


saved   764 rows -> analysis\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\llm_vote_file_level.csv


## Game-level aggregation

Per game: the empirical distribution over the (up to 3) valid stochastic votes
(circle votes included as a real label), its probability mass on the correct
answer (`p_correct`), its entropy, majority diagnostics, and the greedy vote
kept separate.


In [19]:
def vote_label(row: pd.Series) -> str:
    return CIRCLE_VOTE_STRING if row["is_circle_vote"] else row["chosen_player_name"]


def game_level_record(game_id: str, group: pd.DataFrame, gt_row: pd.Series) -> dict:
    valid = group[group["status"].isin(["player_vote", "circle_vote"])]
    stoch = valid[valid["run_label"].isin(STOCHASTIC_RUN_LABELS)]
    greedy = valid[valid["run_label"] == GREEDY_RUN_LABEL]

    labels = [vote_label(r) for _, r in stoch.iterrows()]   # player name or CIRCLE_VOTE_STRING
    counts = Counter(labels)
    n = len(labels)
    dist = {k: v / n for k, v in counts.items()} if n else {}

    p_correct = float(stoch["is_correct"].astype(float).mean()) if n else np.nan
    entropy = float(-(np.array([v for v in dist.values()]) * np.log([v for v in dist.values()])).sum()) if n else np.nan

    majority_vote, has_majority = np.nan, False
    if counts:
        top = max(counts.values())
        winners = [k for k, v in counts.items() if v == top]
        if len(winners) == 1 and top > n / 2:
            majority_vote, has_majority = winners[0], True

    majority_correct = np.nan
    if has_majority:
        mask = [vote_label(r) == majority_vote for _, r in stoch.iterrows()]
        majority_correct = bool(stoch[mask].iloc[0]["is_correct"])

    greedy_vote = vote_label(greedy.iloc[0]) if len(greedy) else None
    greedy_correct = bool(greedy["is_correct"].iloc[0]) if len(greedy) else np.nan

    return {
        "game_id": game_id, "source": gt_row["source"], "session_name": gt_row["session_name"],
        "game_key": gt_row["game_key"],
        "player_names": gt_row["player_names"], "start_roles": gt_row["start_roles"],
        "end_roles": gt_row["end_roles"],
        "has_werewolf": gt_row["has_werewolf"], "correct_vote_type": gt_row["correct_vote_type"],
        "correct_player_names": gt_row["correct_player_names"],
        "n_valid_stochastic": n,
        "stoch_vote_distribution": dist,
        "p_correct": p_correct,
        "stoch_entropy": entropy,
        "at_least_one_correct": bool(stoch["is_correct"].any()) if n else np.nan,
        "stoch_majority_vote": majority_vote, "has_strict_majority": has_majority,
        "stoch_majority_correct": majority_correct,
        "greedy_vote": greedy_vote, "greedy_correct": greedy_correct,
        "greedy_in_stoch_support": (greedy_vote in dist) if greedy_vote is not None else np.nan,
    }


GT_BY_GAME_ID = {row["game_id"]: row for _, row in gt_df.iterrows()}

evaluable = llm_vote_df[llm_vote_df["status"] != "missing_ground_truth"]
llm_vote_game_level = pd.DataFrame([
    game_level_record(gid, grp, GT_BY_GAME_ID[gid]) for gid, grp in evaluable.groupby("game_id")
])
print(f"game-level rows: {len(llm_vote_game_level)}")
display(llm_vote_game_level.head(3))

if SAVE_TABLES:
    save_csv(llm_vote_game_level, MODEL_TABLES_DIR / "llm_vote_game_level.csv",
             json_columns=("player_names", "start_roles", "end_roles",
                           "correct_player_names", "stoch_vote_distribution"))


game-level rows: 191


,game_id,source,session_name,game_key,player_names,start_roles,end_roles,has_werewolf,correct_vote_type,correct_player_names,...,stoch_vote_distribution,p_correct,stoch_entropy,at_least_one_correct,stoch_majority_vote,has_strict_majority,stoch_majority_correct,greedy_vote,greedy_correct,greedy_in_stoch_support
0,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,Ego4D,0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0,Game1,"[Erin, Sebastian, Jack, Brent, Hailey]","[Werewolf, Robber, Moderator, Seer, Troublemaker]","[Robber, Seer, Moderator, Werewolf, Troublemaker]",True,player,[Brent],...,"{'Hailey': 0.6666666666666666, 'No Werewolf': ...",0.000000,0.636514,False,Hailey,True,False,Hailey,False,True
1,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,Ego4D,0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0,Game2,"[Erin, Sebastian, Jack, Brent, Hailey]","[Moderator, Robber, Werewolf, Villager, Villager]","[Moderator, Villager, Werewolf, Villager, Robber]",True,player,[Jack],...,{'No Werewolf': 1.0},0.000000,-0.000000,False,No Werewolf,True,False,No Werewolf,False,True
2,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,Ego4D,0c2659db-7bd4-4b37-9b08-4e247befe382,Game4,"[Elliot, Ashley, James, Chris, Sukeshi, Sian]","[Robber, Werewolf, Insomniac, Seer, Werewolf, ...","[Tanner, Werewolf, Insomniac, Seer, Werewolf, ...",True,player,"[Ashley, Sukeshi]",...,"{'Elliot': 0.3333333333333333, 'Ashley': 0.666...",0.666667,0.636514,True,Ashley,True,True,Sian,False,False


saved   191 rows -> analysis\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\llm_vote_game_level.csv


## Run-level accuracy proxies

Accuracy over valid votes per run, plus status counts. The coherence check
below confirms the probabilistic aggregation: the game-mean of `p_correct`
must equal the mean single-run stochastic accuracy (up to games with missing
runs).


In [20]:
def summarize_run(group: pd.DataFrame) -> pd.Series:
    valid = group[group["status"].isin(["player_vote", "circle_vote"])]
    n_valid = len(valid)
    n_correct = int(valid["is_correct"].sum())
    return pd.Series({
        "decoding": group["decoding"].iloc[0],
        "total_files": len(group),
        "valid_votes": n_valid,
        "player_votes": int((valid["status"] == "player_vote").sum()),
        "circle_votes": int((valid["status"] == "circle_vote").sum()),
        "failed_parses": int((group["status"] == "failed_parse").sum()),
        "invalid_votes": int((group["status"] == "invalid_vote").sum()),
        "missing_ground_truth": int((group["status"] == "missing_ground_truth").sum()),
        "n_correct": n_correct,
        "accuracy": n_correct / n_valid if n_valid else np.nan,
    })


llm_run_summary = evaluable.groupby("run_label").apply(summarize_run, include_groups=False).reset_index()
display(llm_run_summary)

mean_p_correct = llm_vote_game_level["p_correct"].mean()
mean_stoch_acc = llm_run_summary.loc[
    llm_run_summary["run_label"].isin(STOCHASTIC_RUN_LABELS), "accuracy"].mean()
n_incomplete = int((llm_vote_game_level["n_valid_stochastic"] < len(STOCHASTIC_RUN_LABELS)).sum())
print(f"coherence check -- game-mean p_correct: {mean_p_correct:.4f} | "
      f"mean stochastic run accuracy: {mean_stoch_acc:.4f}")
if n_incomplete == 0 and not np.isclose(mean_p_correct, mean_stoch_acc):
    print("WARNING: values differ with no missing runs -- investigate.")
elif n_incomplete > 0:
    print(f"note: {n_incomplete} game(s) have < {len(STOCHASTIC_RUN_LABELS)} valid stochastic votes; "
          "the two means then weight games differently, so a small gap is expected.")

if SAVE_TABLES:
    save_csv(llm_run_summary, MODEL_TABLES_DIR / "llm_run_summary.csv")


,run_label,decoding,total_files,valid_votes,player_votes,circle_votes,failed_parses,invalid_votes,missing_ground_truth,n_correct,accuracy
0,greedy_t0,greedy,191,191,143,48,0,0,0,70,0.366492
1,run_1,stochastic,191,190,151,39,1,0,0,81,0.426316
2,run_2,stochastic,191,189,146,43,2,0,0,77,0.407407
3,run_3,stochastic,191,189,140,49,2,0,0,73,0.386243


coherence check -- game-mean p_correct: 0.4058 | mean stochastic run accuracy: 0.4067
note: 5 game(s) have < 3 valid stochastic votes; the two means then weight games differently, so a small gap is expected.
saved     4 rows -> analysis\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\llm_run_summary.csv


## Printable examples (with justifications)

A few hand-readable cases per category, chosen for relevance to the RQ:
unanimous-correct, unanimous-wrong, split stochastic votes, greedy disagreeing
with the stochastic majority, and circle votes. Saved as JSON + printable text.


In [21]:
def example_category(rec: pd.Series) -> Optional[str]:
    if rec["n_valid_stochastic"] == 0:
        return None
    dist = rec["stoch_vote_distribution"]
    if len(dist) == 1:
        return "unanimous_correct" if rec["p_correct"] == 1.0 else "unanimous_wrong"
    if rec["has_strict_majority"] and rec["greedy_vote"] is not None             and rec["greedy_vote"] != rec["stoch_majority_vote"]:
        return "greedy_vs_majority_disagree"
    if 0 < rec["p_correct"] < 1:
        return "split_votes"
    return "split_votes"


def collect_examples(game_level: pd.DataFrame, file_level: pd.DataFrame) -> list:
    rng = np.random.default_rng(RANDOM_STATE)
    work = game_level.copy()
    work["example_category"] = work.apply(example_category, axis=1)
    # circle votes sampled separately from file level
    examples = []
    for cat, sub in work.dropna(subset=["example_category"]).groupby("example_category"):
        take = sub.sample(min(SAMPLE_PER_EXAMPLE_TYPE, len(sub)), random_state=RANDOM_STATE)
        for _, rec in take.iterrows():
            game_files = file_level[(file_level["game_id"] == rec["game_id"])
                                    & file_level["status"].isin(["player_vote", "circle_vote"])]
            examples.append({
                "category": cat, "game_id": rec["game_id"],
                "correct_player_names": rec["correct_player_names"],
                "p_correct": rec["p_correct"], "greedy_vote": rec["greedy_vote"],
                "votes": [
                    {"run_label": r["run_label"], "vote": vote_label(r),
                     "is_correct": r["is_correct"], "justification": r["justification"]}
                    for _, r in game_files.iterrows()
                ],
            })
    circle_rows = file_level[file_level["is_circle_vote"] == True]
    if len(circle_rows):
        take = circle_rows.sample(min(SAMPLE_PER_EXAMPLE_TYPE, len(circle_rows)), random_state=RANDOM_STATE)
        for _, r in take.iterrows():
            examples.append({
                "category": "circle_vote", "game_id": r["game_id"],
                "correct_player_names": GT_BY_GAME_ID.get(r["game_id"], {}).get("correct_player_names") if r["game_id"] in GT_BY_GAME_ID else None,
                "votes": [{"run_label": r["run_label"], "vote": vote_label(r),
                           "is_correct": r["is_correct"], "justification": r["justification"]}],
            })
    return examples


def render_examples_txt(examples: list) -> str:
    lines = []
    for ex in examples:
        lines.append("=" * 78)
        lines.append(f"[{ex['category']}]  {ex['game_id']}")
        lines.append(f"correct: {ex['correct_player_names']}")
        for v in ex["votes"]:
            lines.append(f"  {v['run_label']:>10}: {v['vote']} (correct={v['is_correct']})")
            if v.get("justification"):
                lines.append(f"      justification: {str(v['justification'])[:500]}")
        lines.append("")
    return "\n".join(lines)


examples = collect_examples(llm_vote_game_level, llm_vote_df)
print(f"{len(examples)} examples across categories:",
      Counter(e["category"] for e in examples))

if SAVE_PRINTABLE_EXAMPLES:
    save_json(examples, MODEL_EXAMPLES_DIR / "vote_examples.json")
    txt_path = MODEL_EXAMPLES_DIR / "vote_examples.txt"
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    txt_path.write_text(render_examples_txt(examples), encoding="utf-8")
    print("saved examples ->", txt_path.relative_to(REPO_ROOT))


15 examples across categories: Counter({'greedy_vs_majority_disagree': 3, 'split_votes': 3, 'unanimous_correct': 3, 'unanimous_wrong': 3, 'circle_vote': 3})
saved examples -> analysis\unsloth_gemma-4-E2B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\examples\vote_examples.txt


## Human outcomes (model-agnostic; saved once outside model folders)

Two tables from the game index alone:

1. `human_game_outcomes.csv` -- the mechanical ONUW table outcome using **all**
   human votes (most-voted player(s) with the >= 2 votes rule, or circle).
2. `village_vote_dispersion.csv` -- votes of **village-aligned players only**
   (alignment from `ALIGNMENT_ROLES_FROM` roles, excluding
   `NON_VILLAGE_ALIGNED_ROLES`): per-game counts, modal target(s), top share,
   entropy, and whether the modal target is an end-Werewolf. Same schema as the
   old model-folder copy, so downstream notebooks only need the new path.

Interpretation caution carried from the thesis framing: these human statistics
are a *rough* difficulty proxy. Humans hold private night information the
models never see, and human votes are an outcome of the same dialogue the
models read -- neither a gold standard nor a pure "social signal".


In [22]:
def shannon_entropy_from_counts(counts: Counter) -> float:
    total = sum(counts.values())
    if total == 0:
        return np.nan
    probs = np.array([c / total for c in counts.values()], dtype=float)
    return float(-(probs * np.log(probs)).sum())


def human_game_outcome(row: pd.Series) -> dict:
    votes = parse_indexed_votes(row["voting_outcome"])
    valid = [v for v in votes if v is not None and 0 <= v < len(row["player_names"])]
    rec = {"game_id": row["game_id"], "source": row["source"], "session_name": row["session_name"],
           "game_key": row["game_key"], "n_players": len(row["player_names"]),
           "n_valid_human_votes": len(valid), "n_missing_human_votes": len(votes) - len(valid),
           "has_werewolf": row["has_werewolf"]}
    if not valid:
        return {**rec, "human_outcome_type": None, "human_target_names": [], "human_outcome_correct": np.nan}
    counts = Counter(valid)
    top = max(counts.values())
    if top < 2:   # ONUW rule: nobody eliminated without >= 2 votes
        return {**rec, "human_outcome_type": "circle", "human_target_names": [],
                "human_outcome_correct": bool(not row["has_werewolf"])}
    targets = sorted(i for i, c in counts.items() if c == top)
    target_names = [row["player_names"][i] for i in targets]
    correct = any(row["end_roles"][i] == "Werewolf" for i in targets)
    return {**rec, "human_outcome_type": "player", "human_target_names": target_names,
            "human_outcome_correct": bool(correct)}


def village_dispersion(row: pd.Series) -> Optional[dict]:
    roles_for_alignment = row["end_roles"] if ALIGNMENT_ROLES_FROM == "end" else row["start_roles"]
    votes = parse_indexed_votes(row["voting_outcome"])
    village_votes = [
        v for voter_idx, v in enumerate(votes)
        if v is not None and 0 <= v < len(row["player_names"])
        and voter_idx < len(roles_for_alignment)
        and roles_for_alignment[voter_idx] not in NON_VILLAGE_ALIGNED_ROLES
    ]
    if not village_votes:
        return None
    counts = Counter(village_votes)
    top = max(counts.values())
    top_targets = sorted(i for i, c in counts.items() if c == top)
    top_names = [row["player_names"][i] for i in top_targets]
    modal_correct = any(row["end_roles"][i] == "Werewolf" for i in top_targets)
    name_counts = {row["player_names"][i]: c for i, c in counts.items()}
    return {
        "game_id": row["game_id"],
        "n_village_aligned_votes": len(village_votes),
        "village_vote_counts": name_counts,
        "village_top_targets": top_targets,
        "village_top_target_names": top_names,
        "village_top_share": top / len(village_votes),
        "village_vote_entropy": shannon_entropy_from_counts(counts),
        "village_modal_correct": bool(modal_correct),
    }


human_game_outcomes = pd.DataFrame([human_game_outcome(r) for _, r in gt_df.iterrows()])
village_rows = [d for _, r in gt_df.iterrows() if (d := village_dispersion(r)) is not None]
village_vote_dispersion = pd.DataFrame(village_rows)

human_summary = pd.DataFrame([{
    "n_games_total": len(gt_df),
    "n_games_with_valid_human_votes": int((human_game_outcomes["n_valid_human_votes"] > 0).sum()),
    "human_outcome_win_rate": float(human_game_outcomes["human_outcome_correct"].mean()),
    "n_games_with_village_votes": len(village_vote_dispersion),
    "village_modal_correct_rate": float(village_vote_dispersion["village_modal_correct"].mean()),
    "alignment_roles_from": ALIGNMENT_ROLES_FROM,
    "non_village_aligned_roles": json.dumps(sorted(NON_VILLAGE_ALIGNED_ROLES)),
}])
display(human_summary)

if SAVE_TABLES:
    save_csv(human_game_outcomes, HUMAN_TABLES_DIR / "human_game_outcomes.csv",
             json_columns=("human_target_names",))
    save_csv(village_vote_dispersion, HUMAN_TABLES_DIR / "village_vote_dispersion.csv",
             json_columns=("village_vote_counts", "village_top_targets", "village_top_target_names"))
    save_csv(human_summary, HUMAN_TABLES_DIR / "human_summary.csv")


,n_games_total,n_games_with_valid_human_votes,human_outcome_win_rate,n_games_with_village_votes,village_modal_correct_rate,alignment_roles_from,non_village_aligned_roles
0,191,185,0.491892,183,0.590164,end,"[""Minion"", ""Tanner"", ""Werewolf""]"


saved   191 rows -> analysis\human_outcomes\prompt_v4\tables\human_game_outcomes.csv
saved   183 rows -> analysis\human_outcomes\prompt_v4\tables\village_vote_dispersion.csv
saved     1 rows -> analysis\human_outcomes\prompt_v4\tables\human_summary.csv


## Next

Notebook 2 (cross-model comparison) reads, per model, `llm_vote_file_level.csv`
and `llm_vote_game_level.csv` from the model folders, and the human tables from
`analysis/human_outcomes/<prompt>/tables/`. 